# TUGAS MANDIRI DATA WRANGLING PERTEMUAN 2
* **Nama Mahasiswa:** Nurul Umaroh
* **NIM:** 25200018

# BAGIAN A
## Analisis Konseptual & Tata Kelola Data

### A.1 Penerjemahan Kebutuhan Bisnis

Perusahaan membutuhkan analisis terhadap transaksi penjualan pada periode
promosi kupon. Informasi yang dibutuhkan meliputi data transaksi, data
pelanggan, status membership pelanggan, serta status pengiriman barang.

Untuk memenuhi kebutuhan tersebut, data diperoleh dari tiga sumber, yaitu:

1. Data transaksi penjualan dari sistem POS dalam format CSV.
2. Data pelanggan dari sistem CRM dalam database SQLite.
3. Data status pengiriman dari sistem logistik melalui REST API dalam format JSON.

Ketiga sumber data tersebut akan diintegrasikan menjadi satu dataset master
untuk keperluan analisis.

### Pemetaan Kebutuhan Bisnis menjadi Kebutuhan Teknis Data

| Kebutuhan Informasi Bisnis | Atribut Teknis | Tipe Data | Sumber Data |
|---|---|---|---|
| Mengidentifikasi transaksi | order_id | String | CSV POS |
| Mengidentifikasi pelanggan | customer_id | String | CSV POS dan SQL CRM |
| Mengetahui tanggal transaksi | order_date | Datetime | CSV POS |
| Mengetahui nilai transaksi | total_amount | Integer/Float | CSV POS |
| Mengetahui penggunaan kupon | coupon_code | String | CSV POS |
| Mengetahui nama pelanggan | customer_name | String | SQL CRM |
| Mengetahui tingkat membership | membership_level | String | SQL CRM |
| Mengetahui status aktif pelanggan | is_active | Integer/Boolean | SQL CRM |
| Mengetahui status pengiriman | delivery_status | String | REST API |

## A.2 Matriks Peran RACI

Matriks RACI digunakan untuk menjelaskan pembagian tanggung jawab
masing-masing pihak dalam proses pengelolaan dan pengolahan data.

| Aktivitas | Head of Business Strategy | Data Management Office / Data Steward | Tim Data Wrangling | Business Intelligence Analyst |
|---|---|---|---|---|
| Menentukan kebutuhan bisnis | A | C | I | R |
| Menentukan standar dan definisi data | C | A/R | C | I |
| Menyiapkan data dari berbagai sumber | I | A | R | C |
| Melakukan data wrangling dan integrasi | I | C | R | C |
| Melakukan validasi kualitas data | I | A | R | C |
| Melakukan analisis bisnis | C | I | C | R/A |
| Menggunakan hasil analisis | A | I | I | R |

### Keterangan RACI

- **R (Responsible)**: pihak yang melaksanakan pekerjaan.
- **A (Accountable)**: pihak yang bertanggung jawab atas hasil akhir.
- **C (Consulted)**: pihak yang memberikan masukan atau konsultasi.
- **I (Informed)**: pihak yang perlu mendapatkan informasi mengenai kegiatan tersebut.

In [19]:
import pandas as pd
import sqlite3
import json
import os

print("Library berhasil di-import")

# Membuat data transaksi penjualan
data_transaksi = {
    "order_id": [
        "ORD001", "ORD002", "ORD003", "ORD004", "ORD005",
        "ORD006", "ORD007", "ORD008", "ORD009", "ORD010"
    ],

    "customer_id": [
        "C001", "C002", "C003", "C004", "C005",
        "C006", "C007", "C008", "C009", "C010"
    ],

    "order_date": [
        "2026-09-01",
        "2026-09-02",
        "2026-09-03",
        "2026-09-04",
        "2026-09-05",
        "2026-09-06",
        "2026-09-07",
        "2026-09-08",
        "2026-09-09",
        "2026-09-10"
    ],

    "total_amount": [
        150000,
        200000,
        125000,
        300000,
        175000,
        250000,
        180000,
        220000,
        350000,
        190000
    ],

    "coupon_code": [
        "DISC10",
        "DISC20",
        "DISC10",
        "DISC30",
        "DISC10",
        "DISC20",
        "DISC10",
        "DISC20",
        "DISC30",
        "DISC10"
    ]
}

df_transaksi_dummy = pd.DataFrame(data_transaksi)

# Simpan menjadi CSV
df_transaksi_dummy.to_csv(
    "transaksi_penjualan.csv",
    index=False
)

print("File transaksi_penjualan.csv berhasil dibuat!")

display(df_transaksi_dummy)

Library berhasil di-import
File transaksi_penjualan.csv berhasil dibuat!


,order_id,customer_id,order_date,total_amount,coupon_code
0,ORD001,C001,2026-09-01,150000,DISC10
1,ORD002,C002,2026-09-02,200000,DISC20
2,ORD003,C003,2026-09-03,125000,DISC10
3,ORD004,C004,2026-09-04,300000,DISC30
4,ORD005,C005,2026-09-05,175000,DISC10
5,ORD006,C006,2026-09-06,250000,DISC20
6,ORD007,C007,2026-09-07,180000,DISC10
7,ORD008,C008,2026-09-08,220000,DISC20
8,ORD009,C009,2026-09-09,350000,DISC30
9,ORD010,C010,2026-09-10,190000,DISC10


In [23]:
# Membuat data pelanggan CRM

data_customer = {
    "customer_id": [
        "C001", "C002", "C003", "C004", "C005",
        "C006", "C007", "C008", "C009", "C010"
    ],

    "customer_name": [
        "Andi",
        "Budi",
        "Citra",
        "Dinda",
        "Eka",
        "Fajar",
        "Gita",
        "Hani",
        "Indra",
        "Joko"
    ],

    "membership_level": [
        "Gold",
        "Silver",
        "Gold",
        "Bronze",
        "Gold",
        "Silver",
        "Gold",
        "Silver",
        "Gold",
        "Bronze"
    ],

    "is_active": [
        1,
        1,
        1,
        0,
        1,
        1,
        1,
        0,
        1,
        1
    ]
}

df_customer_dummy = pd.DataFrame(data_customer)

# Membuat database SQLite
conn = sqlite3.connect("crm_database.db")

# Memasukkan data ke tabel customer
df_customer_dummy.to_sql(
    "customer",
    conn,
    if_exists="replace",
    index=False
)

conn.close()

print("Database crm_database.db berhasil dibuat!")

display(df_customer_dummy)

Database crm_database.db berhasil dibuat!


,customer_id,customer_name,membership_level,is_active
0,C001,Andi,Gold,1
1,C002,Budi,Silver,1
2,C003,Citra,Gold,1
3,C004,Dinda,Bronze,0
4,C005,Eka,Gold,1
5,C006,Fajar,Silver,1
6,C007,Gita,Gold,1
7,C008,Hani,Silver,0
8,C009,Indra,Gold,1
9,C010,Joko,Bronze,1


In [24]:
# Membuat data logistik dalam bentuk JSON nested

data_logistik = {
    "data": [
        {
            "order_id": "ORD001",
            "shipping": {
                "status": "Delivered"
            }
        },
        {
            "order_id": "ORD002",
            "shipping": {
                "status": "On Delivery"
            }
        },
        {
            "order_id": "ORD003",
            "shipping": {
                "status": "Delivered"
            }
        },
        {
            "order_id": "ORD004",
            "shipping": {
                "status": "Processing"
            }
        },
        {
            "order_id": "ORD005",
            "shipping": {
                "status": "Delivered"
            }
        },
        {
            "order_id": "ORD006",
            "shipping": {
                "status": "On Delivery"
            }
        },
        {
            "order_id": "ORD007",
            "shipping": {
                "status": "Delivered"
            }
        },
        {
            "order_id": "ORD008",
            "shipping": {
                "status": "Processing"
            }
        },
        {
            "order_id": "ORD009",
            "shipping": {
                "status": "Delivered"
            }
        },
        {
            "order_id": "ORD010",
            "shipping": {
                "status": "On Delivery"
            }
        }
    ]
}

# Simpan menjadi file JSON
with open("logistik.json", "w") as file:
    json.dump(data_logistik, file, indent=4)

print("File logistik.json berhasil dibuat!")

File logistik.json berhasil dibuat!


In [25]:
print("File yang tersedia:")

for file in os.listdir():
    if file.endswith((".csv", ".db", ".json")):
        print("-", file)

File yang tersedia:
- crm_loyalty.db
- logistik.json
- transaksi_pos.csv
- transaksi_penjualan.csv
- crm_database.db
- kamus_data_hasil.csv


In [ ]:
# B.1 Ekstraksi Data Transaksi dari File CSV

import pandas as pd

df_transaksi = pd.read_csv(
    "transaksi_penjualan.csv",
    dtype={"order_id": str}
)

# Mengubah kolom order_date menjadi tipe datetime
df_transaksi["order_date"] = pd.to_datetime(
    df_transaksi["order_date"]
)

# Menampilkan data
display(df_transaksi)

### Penjelasan B.1

Data transaksi penjualan berhasil dimuat dari file CSV `transaksi_penjualan.csv`.
Kolom `order_id` dipaksa bertipe string menggunakan parameter `dtype`, sedangkan
kolom `order_date` dikonversi menjadi tipe data datetime menggunakan
`pd.to_datetime()`. Dengan demikian, data transaksi siap digunakan untuk
tahap pengolahan dan integrasi dengan sumber data lainnya.

In [28]:
# B.1 Ekstraksi Data Transaksi dari File CSV

import pandas as pd

df_transaksi = pd.read_csv(
    "transaksi_penjualan.csv",
    dtype={"order_id": str}
)

# Mengubah kolom order_date menjadi tipe datetime
df_transaksi["order_date"] = pd.to_datetime(
    df_transaksi["order_date"]
)

# Menampilkan data
display(df_transaksi)

,order_id,customer_id,order_date,total_amount,coupon_code
0,ORD001,C001,2026-09-01,150000,DISC10
1,ORD002,C002,2026-09-02,200000,DISC20
2,ORD003,C003,2026-09-03,125000,DISC10
3,ORD004,C004,2026-09-04,300000,DISC30
4,ORD005,C005,2026-09-05,175000,DISC10
5,ORD006,C006,2026-09-06,250000,DISC20
6,ORD007,C007,2026-09-07,180000,DISC10
7,ORD008,C008,2026-09-08,220000,DISC20
8,ORD009,C009,2026-09-09,350000,DISC30
9,ORD010,C010,2026-09-10,190000,DISC10


In [29]:
print("Tipe data setiap kolom:")
print(df_transaksi.dtypes)

Tipe data setiap kolom:
order_id                object
customer_id             object
order_date      datetime64[ns]
total_amount             int64
coupon_code             object
dtype: object


### Penjelasan B.2

Data pelanggan diekstrak dari database SQLite menggunakan koneksi
`sqlite3`. Query SQL digunakan untuk mengambil data pelanggan dari tabel
`customer`. Filter `WHERE is_active = 1` diterapkan agar hanya pelanggan
yang berstatus aktif yang digunakan dalam proses analisis.

In [30]:
# B.2 Koneksi ke Database CRM

import sqlite3

conn = sqlite3.connect("crm_database.db")

print("Berhasil terhubung ke database CRM")

Berhasil terhubung ke database CRM


In [31]:
query_customer = """
SELECT
    customer_id,
    customer_name,
    membership_level,
    is_active
FROM customer
WHERE is_active = 1
"""

print(query_customer)


SELECT
    customer_id,
    customer_name,
    membership_level,
    is_active
FROM customer
WHERE is_active = 1



In [33]:
df_customer = pd.read_sql(
    query_customer,
    conn
)

display(df_customer)

,customer_id,customer_name,membership_level,is_active
0,C001,Andi,Gold,1
1,C002,Budi,Silver,1
2,C003,Citra,Gold,1
3,C005,Eka,Gold,1
4,C006,Fajar,Silver,1
5,C007,Gita,Gold,1
6,C009,Indra,Gold,1
7,C010,Joko,Bronze,1


In [34]:
conn.close()

print("Koneksi database ditutup.")

Koneksi database ditutup.


### Penjelasan B.3

Data logistik menggunakan struktur JSON sebagai simulasi response dari
REST API. Data JSON memiliki struktur nested pada bagian `shipping.status`.

Struktur nested tersebut dinormalisasi menggunakan fungsi
`pd.json_normalize()` sehingga data dapat digunakan dalam bentuk dataframe.

Hasil normalisasi disimpan dalam dataframe `df_logistik`, kemudian kolom
`shipping.status` diubah menjadi `delivery_status` agar lebih mudah digunakan
pada proses integrasi data.

In [40]:
# B.3 REST API & Normalisasi JSON

import json
import pandas as pd

# Membaca data JSON logistik
with open("logistik.json", "r") as file:
    data_json = json.load(file)

print("Data JSON berhasil dibaca")
print(data_json)

Data JSON berhasil dibaca
{'data': [{'order_id': 'ORD001', 'shipping': {'status': 'Delivered'}}, {'order_id': 'ORD002', 'shipping': {'status': 'On Delivery'}}, {'order_id': 'ORD003', 'shipping': {'status': 'Delivered'}}, {'order_id': 'ORD004', 'shipping': {'status': 'Processing'}}, {'order_id': 'ORD005', 'shipping': {'status': 'Delivered'}}, {'order_id': 'ORD006', 'shipping': {'status': 'On Delivery'}}, {'order_id': 'ORD007', 'shipping': {'status': 'Delivered'}}, {'order_id': 'ORD008', 'shipping': {'status': 'Processing'}}, {'order_id': 'ORD009', 'shipping': {'status': 'Delivered'}}, {'order_id': 'ORD010', 'shipping': {'status': 'On Delivery'}}]}


In [41]:
# Flatten / normalisasi JSON

df_logistik = pd.json_normalize(
    data_json["data"]
)

display(df_logistik)

,order_id,shipping.status
0,ORD001,Delivered
1,ORD002,On Delivery
2,ORD003,Delivered
3,ORD004,Processing
4,ORD005,Delivered
5,ORD006,On Delivery
6,ORD007,Delivered
7,ORD008,Processing
8,ORD009,Delivered
9,ORD010,On Delivery


In [42]:
df_logistik = df_logistik.rename(
    columns={
        "shipping.status": "delivery_status"
    }
)

display(df_logistik)

,order_id,delivery_status
0,ORD001,Delivered
1,ORD002,On Delivery
2,ORD003,Delivered
3,ORD004,Processing
4,ORD005,Delivered
5,ORD006,On Delivery
6,ORD007,Delivered
7,ORD008,Processing
8,ORD009,Delivered
9,ORD010,On Delivery


In [43]:
# Melakukan normalisasi JSON

df_logistik = pd.json_normalize(
    data_json["data"]
)

display(df_logistik)

,order_id,shipping.status
0,ORD001,Delivered
1,ORD002,On Delivery
2,ORD003,Delivered
3,ORD004,Processing
4,ORD005,Delivered
5,ORD006,On Delivery
6,ORD007,Delivered
7,ORD008,Processing
8,ORD009,Delivered
9,ORD010,On Delivery


In [44]:
df_logistik = df_logistik.rename(
    columns={
        "shipping.status": "delivery_status"
    }
)

display(df_logistik)

,order_id,delivery_status
0,ORD001,Delivered
1,ORD002,On Delivery
2,ORD003,Delivered
3,ORD004,Processing
4,ORD005,Delivered
5,ORD006,On Delivery
6,ORD007,Delivered
7,ORD008,Processing
8,ORD009,Delivered
9,ORD010,On Delivery


### Alasan Penggunaan LEFT JOIN

LEFT JOIN digunakan karena data transaksi penjualan dari POS merupakan data
utama dalam analisis. Seluruh transaksi harus tetap dipertahankan meskipun
informasi pelanggan atau status pengiriman tidak ditemukan pada sumber data
lain.

Dengan LEFT JOIN, transaksi yang tidak memiliki pasangan pada data CRM tetap
dipertahankan dan nilai yang tidak tersedia akan ditampilkan sebagai missing
value. Hal ini memungkinkan missing value tersebut diperiksa pada tahap
validasi data.

In [46]:
df_master_analisis = pd.merge(
    df_transaksi,
    df_customer,
    on="customer_id",
    how="left"
)

display(df_master_analisis)

,order_id,customer_id,order_date,total_amount,coupon_code,customer_name,membership_level,is_active
0,ORD001,C001,2026-09-01,150000,DISC10,Andi,Gold,1.0
1,ORD002,C002,2026-09-02,200000,DISC20,Budi,Silver,1.0
2,ORD003,C003,2026-09-03,125000,DISC10,Citra,Gold,1.0
3,ORD004,C004,2026-09-04,300000,DISC30,NaN,NaN,NaN
4,ORD005,C005,2026-09-05,175000,DISC10,Eka,Gold,1.0
5,ORD006,C006,2026-09-06,250000,DISC20,Fajar,Silver,1.0
6,ORD007,C007,2026-09-07,180000,DISC10,Gita,Gold,1.0
7,ORD008,C008,2026-09-08,220000,DISC20,NaN,NaN,NaN
8,ORD009,C009,2026-09-09,350000,DISC30,Indra,Gold,1.0
9,ORD010,C010,2026-09-10,190000,DISC10,Joko,Bronze,1.0


In [47]:
how="left"

In [48]:
df_master_analisis = pd.merge(
    df_master_analisis,
    df_logistik,
    on="order_id",
    how="left"
)

display(df_master_analisis)

,order_id,customer_id,order_date,total_amount,coupon_code,customer_name,membership_level,is_active,delivery_status
0,ORD001,C001,2026-09-01,150000,DISC10,Andi,Gold,1.0,Delivered
1,ORD002,C002,2026-09-02,200000,DISC20,Budi,Silver,1.0,On Delivery
2,ORD003,C003,2026-09-03,125000,DISC10,Citra,Gold,1.0,Delivered
3,ORD004,C004,2026-09-04,300000,DISC30,NaN,NaN,NaN,Processing
4,ORD005,C005,2026-09-05,175000,DISC10,Eka,Gold,1.0,Delivered
5,ORD006,C006,2026-09-06,250000,DISC20,Fajar,Silver,1.0,On Delivery
6,ORD007,C007,2026-09-07,180000,DISC10,Gita,Gold,1.0,Delivered
7,ORD008,C008,2026-09-08,220000,DISC20,NaN,NaN,NaN,Processing
8,ORD009,C009,2026-09-09,350000,DISC30,Indra,Gold,1.0,Delivered
9,ORD010,C010,2026-09-10,190000,DISC10,Joko,Bronze,1.0,On Delivery


In [49]:
print("Jumlah baris:", len(df_master_analisis))
print("Jumlah kolom:", len(df_master_analisis.columns))

print("\nNama kolom:")
print(df_master_analisis.columns.tolist())

display(df_master_analisis)

Jumlah baris: 10
Jumlah kolom: 9

Nama kolom:
['order_id', 'customer_id', 'order_date', 'total_amount', 'coupon_code', 'customer_name', 'membership_level', 'is_active', 'delivery_status']


,order_id,customer_id,order_date,total_amount,coupon_code,customer_name,membership_level,is_active,delivery_status
0,ORD001,C001,2026-09-01,150000,DISC10,Andi,Gold,1.0,Delivered
1,ORD002,C002,2026-09-02,200000,DISC20,Budi,Silver,1.0,On Delivery
2,ORD003,C003,2026-09-03,125000,DISC10,Citra,Gold,1.0,Delivered
3,ORD004,C004,2026-09-04,300000,DISC30,NaN,NaN,NaN,Processing
4,ORD005,C005,2026-09-05,175000,DISC10,Eka,Gold,1.0,Delivered
5,ORD006,C006,2026-09-06,250000,DISC20,Fajar,Silver,1.0,On Delivery
6,ORD007,C007,2026-09-07,180000,DISC10,Gita,Gold,1.0,Delivered
7,ORD008,C008,2026-09-08,220000,DISC20,NaN,NaN,NaN,Processing
8,ORD009,C009,2026-09-09,350000,DISC30,Indra,Gold,1.0,Delivered
9,ORD010,C010,2026-09-10,190000,DISC10,Joko,Bronze,1.0,On Delivery


### Penjelasan C.1

Kamus data dibuat secara otomatis menggunakan function
`buat_data_dictionary()`. Function tersebut menghasilkan informasi
metadata untuk setiap kolom pada `df_master_analisis`, meliputi nama kolom,
tipe data, jumlah dan persentase missing value, jumlah nilai unik, serta
sampel nilai.

Hasil kamus data kemudian disimpan dalam file `kamus_data_hasil.csv`.

In [50]:
# C.1 Otomasi Kamus Data

def buat_data_dictionary(df):

    dictionary = pd.DataFrame({
        "Nama Kolom": df.columns,

        "Tipe Data": [
            str(df[col].dtype)
            for col in df.columns
        ],

        "Jumlah Missing": [
            df[col].isna().sum()
            for col in df.columns
        ],

        "Persentase Missing": [
            df[col].isna().mean() * 100
            for col in df.columns
        ],

        "Jumlah Nilai Unik": [
            df[col].nunique()
            for col in df.columns
        ],

        "Sampel Nilai": [
            df[col].dropna().head(3).tolist()
            for col in df.columns
        ]
    })

    return dictionary

In [51]:
df_dictionary = buat_data_dictionary(
    df_master_analisis
)

display(df_dictionary)

,Nama Kolom,Tipe Data,Jumlah Missing,Persentase Missing,Jumlah Nilai Unik,Sampel Nilai
0,order_id,object,0,0.0,10,"[ORD001, ORD002, ORD003]"
1,customer_id,object,0,0.0,10,"[C001, C002, C003]"
2,order_date,datetime64[ns],0,0.0,10,"[2026-09-01 00:00:00, 2026-09-02 00:00:00, 202..."
3,total_amount,int64,0,0.0,10,"[150000, 200000, 125000]"
4,coupon_code,object,0,0.0,3,"[DISC10, DISC20, DISC10]"
5,customer_name,object,2,20.0,8,"[Andi, Budi, Citra]"
6,membership_level,object,2,20.0,3,"[Gold, Silver, Gold]"
7,is_active,float64,2,20.0,1,"[1.0, 1.0, 1.0]"
8,delivery_status,object,0,0.0,3,"[Delivered, On Delivery, Delivered]"


In [52]:
df_dictionary.to_csv(
    "kamus_data_hasil.csv",
    index=False
)

print("kamus_data_hasil.csv berhasil dibuat!")

kamus_data_hasil.csv berhasil dibuat!


### Penjelasan C.2

Validasi kualitas dan integritas data dilakukan menggunakan automated
assertions.

Tiga aturan bisnis yang divalidasi adalah:

1. `order_id` tidak boleh memiliki duplikasi karena digunakan sebagai
   primary key transaksi.
2. `total_amount` tidak boleh memiliki nilai negatif.
3. `order_date` tidak boleh berada pada tanggal setelah hari ini.

Seluruh assertion berhasil dijalankan tanpa menghasilkan error sehingga
data memenuhi ketiga aturan validasi tersebut.

In [53]:
# C.2.1 Validasi duplikasi primary key

assert not df_master_analisis["order_id"].duplicated().any(), \
    "ERROR: Terdapat duplikasi pada order_id!"

print("VALIDASI 1 BERHASIL: Tidak ada duplikasi order_id.")

VALIDASI 1 BERHASIL: Tidak ada duplikasi order_id.


In [54]:
# C.2.2 Validasi nilai total_amount

assert (df_master_analisis["total_amount"] >= 0).all(), \
    "ERROR: Terdapat total_amount bernilai negatif!"

print("VALIDASI 2 BERHASIL: Tidak ada total_amount negatif.")

VALIDASI 2 BERHASIL: Tidak ada total_amount negatif.


In [55]:
# C.2.3 Validasi tanggal transaksi

hari_ini = pd.Timestamp.today().normalize()

assert (
    df_master_analisis["order_date"] <= hari_ini
).all(), \
    "ERROR: Terdapat order_date di masa depan!"

print("VALIDASI 3 BERHASIL: Tidak ada order_date di masa depan.")

VALIDASI 3 BERHASIL: Tidak ada order_date di masa depan.
